# RiskLens 02 · Ask Nemotron why the allowance moved

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luizkauffmann/Nvidia-Redhat-RiskLens/blob/main/notebooks/02_Agent_Investigator.ipynb)

An **agent on top of the booked CECL runs**. Nemotron 3.5 Lightning (30B total / 3B active, within the 33B limit) plans the investigation and calls RiskLens's deterministic tools. A grounding check verifies that every number in its answer came from a tool result, and each answer is scored with the **same verifier the NeMo Gym environment uses**.

**Local or Colab.** Secrets come from `.env` locally or from Colab Secrets (key icon) in Colab: `NVIDIA_API_KEY`, `RISKLENS_VLLM_URL`. Colab's GPUs are too small for Nemotron, so in Colab use `nvidia` or point `vllm` at your Brev box. Pick the model in the backend cell:

| Backend | What you need |
|---|---|
| `vllm` (default) | Nemotron served by vLLM / Red Hat AI on this machine or a Brev box: `bash deploy/brev_serve_nemotron.sh`, then `RISKLENS_VLLM_URL` if it isn't `localhost:8000` |
| `nvidia` | `NVIDIA_API_KEY=nvapi-...` in `.env` (free key from build.nvidia.com) |
| `mock` | nothing: deterministic planner, for testing without a model |

## 1 · Setup

In [ ]:
# RiskLens bootstrap: finds the repo checkout (local Jupyter) or clones it (Colab)
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Luizkauffmann/Nvidia-Redhat-RiskLens.git"

def _find_repo():
    env = os.environ.get("RISKLENS_REPO")
    for p in ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]:
        if (p / "config" / "gym.yaml").exists() and (p / "src" / "tools").exists():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    if not IN_COLAB:
        raise SystemExit("Start Jupyter from the Nvidia-Redhat-RiskLens checkout (or set RISKLENS_REPO).")
    REPO_DIR = Path("/content/Nvidia-Redhat-RiskLens")
    if (REPO_DIR / ".git").exists():                 # runtime restarted: refresh the existing clone
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt"), "matplotlib"], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.utils import config
config.reload()
from src.utils.notebook import M, ensure_demo_data, endpoint_up, load_secrets, secret
print("secrets:", load_secrets() or "none (.env locally, Colab Secrets in Colab)")
print("repo   :", REPO_DIR, "| python", sys.version.split()[0], "| colab" if IN_COLAB else "| local")
print("data   :", ensure_demo_data())

## 2 · Connect to the model

In [ ]:
# Which model answers. Local default: Nemotron 3.5 Lightning on vLLM (deploy/brev_serve_nemotron.sh).
#   vllm   -> RISKLENS_VLLM_URL (default http://localhost:8000/v1; a Brev box works too)
#   nvidia -> NVIDIA API catalog, needs NVIDIA_API_KEY in .env
#   mock   -> deterministic planner, no model (offline demo / CI)
#   auto   -> vllm if it answers, else nvidia if a key is set, else mock
BACKEND = os.environ.get("RISKLENS_AGENT_BACKEND", "auto")
VLLM_URL = os.environ.get("RISKLENS_VLLM_URL", "http://localhost:8000/v1")
ENABLE_THINKING = True                      # Nemotron reasoning mode: slower, plans better

if BACKEND == "auto":
    BACKEND = "vllm" if endpoint_up(VLLM_URL) else ("nvidia" if secret("NVIDIA_API_KEY") else "mock")
overrides = {"provider": BACKEND, "enable_thinking": ENABLE_THINKING}
if BACKEND == "vllm":
    overrides["base_url"] = VLLM_URL
    overrides["model"] = os.environ.get("RISKLENS_VLLM_MODEL", "nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4")
elif BACKEND == "nvidia":
    os.environ["NVIDIA_API_KEY"] = secret("NVIDIA_API_KEY") or ""

from src.llm.provider import get_provider
provider = get_provider(overrides)
print("backend:", BACKEND, "|", provider.describe())

In [ ]:
# Smoke test: one plain turn + one tool call
from src.tools import tool_schemas
r = provider.chat([{"role": "user", "content": "Reply with exactly: RiskLens ready"}])
print(f"chat  : {(r.content or '').strip()[:60]!r}  ({r.latency_ms:.0f} ms)")
tools = [s for s in tool_schemas() if s["function"]["name"] == "compare_allowance"]
r = provider.chat([{"role": "user", "content": "How much did the CECL allowance change between 2026-07-31 and 2026-08-31?"}], tools=tools)
print("tools :", [(t.name, t.arguments) for t in r.tool_calls] or (r.content or "")[:200], "| format:", r.raw_tool_format)

## 3 · Ask
`ask()` runs one investigation and shows the answer, a grounding badge, the **Gym reward** when the question is in the task bank, and the trace: every model turn and tool call with its parameters and `call_id`.

In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display
from src.agents.orchestrator import build_orchestrator
from src.evaluation import verifier
from src.evaluation.tasks import build_tasks
from src.tools import ToolCallLog

LOG = ToolCallLog(config.path("outputs") / "agent_tool_calls.jsonl")
agent = build_orchestrator(provider, log=LOG)
TASKS = {t.question: t.to_dict() for t in build_tasks()}
RUNS = []

def ask(question, show_trace=True):
    run = agent.run(question)
    RUNS.append(run)
    g = (run.grounding or {}).get("status", "n/a")
    badge = {"PASS": "✅ every number traced to a tool result", "NO_NUMBERS": "✅ no figures quoted",
             "FAIL": "⚠️ unsupported numbers: " + ", ".join(u["text"] for u in (run.grounding or {}).get("ungrounded", []))}.get(g, g)
    score = ""
    if question in TASKS:
        res = verifier.score(*verifier.from_agent_run(run), TASKS[question])
        run.events.append({"type": "gym_reward", **res.to_dict()})
        missed = [c["desc"] for c in res.checks if not c["passed"]]
        score = f" · Gym reward **{res.reward:.2f}**" + (f" (missed: {', '.join(missed)})" if missed else "")
    display(Markdown(f"**Q: {question}**\n\n{run.answer}\n\n---\n*{run.provider.get('model')} · {run.status} · "
                     f"{run.duration_s}s · {len(run.tool_records)} tool calls · grounding: {badge}*{score}"))
    if show_trace:
        rows = []
        for e in run.events:
            if e["type"] == "llm":
                rows.append({"step": "model", "detail": ", ".join(e["tool_calls"]) or "final answer", "status": e["tool_format"], "ms": e["latency_ms"]})
            elif e["type"] == "tool":
                rows.append({"step": "tool", "detail": f"{e['name']}({json.dumps(e.get('parameters') or {})})",
                             "status": "duplicate" if e.get("duplicate") else e["status"], "ms": e.get("duration_ms")})
                if e.get("error"):
                    rows.append({"step": "  hint", "detail": f"{e['error']['message']} -> {e['error'].get('hint')}", "status": "", "ms": None})
            elif e["type"] == "grounding":
                rows.append({"step": "grounding", "detail": ", ".join(e["ungrounded"]) or "all numbers supported", "status": e["status"], "ms": None})
        with pd.option_context("display.max_colwidth", 140):
            display(pd.DataFrame(rows))
    return run

### The demo questions

In [ ]:
_ = ask("Why did our allowance increase in August? What were the major drivers?")

In [ ]:
_ = ask("Which macroeconomic variable had the largest impact on the allowance?")

In [ ]:
_ = ask("What happened specifically in Commercial Real Estate?")

In [ ]:
_ = ask("Which borrowers contributed most to the increase? Who should I investigate first?")

In [ ]:
_ = ask("Could this increase just be because we changed models?")

In [ ]:
_ = ask("Does the attribution reconcile to the total allowance change?")

### Your own question

In [ ]:
_ = ask("Explain borrower CP_040408: what changed and how much did it add to the allowance?")

## 4 · Tool-calling scorecard
Reliability metrics for the **Tool-Calling** track: success rate, repaired arguments, recovered errors, tool calls written as text, grounding, and the Gym reward where the question is in the task bank.

In [ ]:
def gym_reward(run):
    e = next((e for e in run.events if e["type"] == "gym_reward"), None)
    return e["reward"] if e else None

summary = pd.DataFrame([{"question": r.question[:60], "gym_reward": gym_reward(r), **r.metrics()} for r in RUNS])
display(summary[["question", "gym_reward", "tool_calls", "tool_errors", "recovered_after_error", "repaired_arguments",
                 "text_fallback_tool_calls", "revisions", "grounding", "llm_turns", "total_tokens", "duration_s"]])
LOG.stats()

## 5 · The RiskLens team (multi-agent)
The same model now plays several roles. The **orchestrator** delegates to specialists, each with its own prompt and a short tool list (`config/agents.yaml`):

| Specialist | Owns |
|---|---|
| Portfolio, Credit, Macro, Model Risk | exposure; borrower deterioration; scenarios and macro variables; models, weights, overlays (Q-factors) |
| Reconciliation | does the explanation add up (PASS / WARNING / FAIL) |
| **Quant Analyst** | custom questions: writes pandas code that runs in a read-only sandbox (no network, no writes, no ground truth) |
| **Reporting Analyst** | the Excel attribution pack: live formulas, a Sources sheet with every tool call |

One base model for every role keeps the whole system within the 33B-parameter limit. Specialist findings carry their tool results, so the orchestrator's grounding check still traces every number.

In [ ]:
from src.agents.team import build_team

team = build_team(provider, log=LOG)

def ask_team(question):
    run = team.run(question)
    RUNS.append(run)
    sub = next((e for e in run.events if e["type"] == "team"), {"specialists": [], "sub_runs": []})
    g = (run.grounding or {}).get("status", "n/a")
    display(Markdown(f"**Q: {question}**\n\n{run.answer}\n\n---\n*specialists: {', '.join(sub['specialists']) or 'none'} · "
                     f"orchestrator tools: {', '.join(run.tools_used)} · grounding: {g} · {run.duration_s}s*"))
    rows = [{"agent": s["agent"], "tools": ", ".join(t["tool_name"] for t in s["tool_calls"]),
             "grounding": (s.get("grounding") or {}).get("status"), "seconds": s["duration_s"]} for s in sub["sub_runs"]]
    if rows:
        display(pd.DataFrame(rows))
    return run

_ = ask_team("Why did the allowance increase in August, and which borrowers should I look at first?")

### The Quant Analyst: a question no fixed tool answers
With a real model, the quant writes pandas code; its hash and status show in the trace. (The offline `mock` planner doesn't write code, so it only lists the datasets.)

In [ ]:
run = ask_team("Using Python: which 5 states carry the most CECL ECL in August 2026, and how much did each change since July?")
for s in next(e for e in run.events if e["type"] == "team")["sub_runs"]:
    for t in s["tool_calls"]:
        if t["tool_name"] == "run_python_analysis":
            print("code ran: sha256", t["result_sha256"], "| status", t["status"])

### The Reporting Analyst: the Excel attribution pack

In [ ]:
run = ask_team("Build the Excel attribution pack for the ALLL committee.")
packs = sorted((config.path("outputs") / "reports").glob("*.xlsx"))
print("workbooks:", [p.name for p in packs])
if IN_COLAB and packs:
    from google.colab import files
    files.download(str(packs[-1]))                 # Colab: download to your laptop